# Performance optimization with Vectorized Python UDFs

In this notebook we will explore how to make Python UDFs faster with Vectorized Python UDFs.  (Documentation: https://docs.snowflake.com/en/developer-guide/udf/python/udf-python-batch)

In the previous notebook, the last step predicted churn across a batch of rows (customers).  When we call the ```predict_churn()``` Python UDF, Snowflake invokes the UDF once per row, adding significant overhead to processing the entire batch.  Instead, we can request that Snowflake pass batches of rows to the UDF instead of one row at a time, reducing the number of UDF invocations required to process the entire batch of rows.

To focus on vectorization, we will use the same classifier developed in the previous notebook to produce a before-vectorization and after-vectorization benchmark on Python UDFs.

Some code changes will be needed to our function, namely our function must accept this batch in the form of a pandas dataframe, and return a batch of results as a pandas dataframe.

<strong>NOTE:</strong>
The Snowpark dataframe API is not being used to vectorize.  In this notebook we will only be using Snowpark to serialize and create the new UDF in Snowflake, but the UDF itself will be utilizing Pandas dataframes for Input/Output with the Vectorized Python UDF API.

### Steps below:

1. Connect to Snowflake
2. Prepare a larger set of customers to score
3. Establish baseline timing, non-vectorized Python UDF
4. Create a vectorized version of the UDF
5. Obtain timing with same batch, new Vectorized Python UDF

---
#### 1. Connect to Snowflake

In [ ]:
config_dir = '/home/jovyan/.ssh'
configfile = config_dir + '/sf_config'

In [ ]:
import snowflake.snowpark
from snowflake.snowpark.functions import * 
from snowflake.snowpark.session import Session
from snowflake.snowpark.types import *

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn import metrics

import pandas as pd
import sys

In [ ]:
# Load configuration file
with open(configfile) as f:
    lines = f.readlines()
    
# Convert configuration to a properties map
props = {}
for line in lines:
    (key, value) = line.split('=')
    props.update({key.lower() : value[0:-1]})
    
# Convert the private key to a DER-encoded bytes object
from cryptography.hazmat.primitives import serialization
from cryptography.hazmat.backends import default_backend

with open(props['private_key_file'], 'rb') as key:
    private_key = serialization.load_pem_private_key(
        key.read(),
        password=None,
        backend=default_backend()
    )
    
private_key_bytes = private_key.private_bytes(
    encoding=serialization.Encoding.DER,
    format=serialization.PrivateFormat.PKCS8,
    encryption_algorithm=serialization.NoEncryption()
)

# Connect to Snowflake
session = Session.builder.configs({**props, **{'private_key': private_key_bytes}}).create()

Confirm connection and the session's Snowflake context.

In [ ]:
print( session.get_current_role() , session.get_current_warehouse() ,session.get_current_database() ,session.get_current_schema())

#### 2. Prepare a larger set of customers to score

In [ ]:
# big_customers will be our larger set of customers to score
session.sql('''CREATE OR REPLACE TABLE big_customers AS
            SELECT
                CUSTOMER_ID,
                SURNAME_MASKED,
                CREDIT_SCORE,
                GEOGRAPHY,
                GENDER,
                AGE,
                TENURE,
                MILEAGE_POINTS,
                NUM_OF_PRODUCTS,
                HAS_AIRLINE_CREDIT_CARD,
                IS_ACTIVE_MEMBER,
                ESTIMATED_SALARY
            FROM data_science_db.public.customer_churn
            ''').show()

session.sql("INSERT INTO big_customers SELECT * FROM big_customers").show()
session.sql("INSERT INTO big_customers SELECT * FROM big_customers").show()
session.sql("INSERT INTO big_customers SELECT * FROM big_customers").show()

# each successive INSERT doubles up the big_customers table

# Verify the size of this new table is 80,000 rows
session.table("big_customers").count()

#### 3. Establish baseline timing, non-vectorized Python UDF

In [ ]:
%%time

# We will use the UDF created in the previous notebook as our baseline
# This should take less than 2 minutes on an x-small warehouse
session.sql('''
    create or replace temporary table predictions_1 as
    select customer_id, 
           surname_masked,
           predict_churn(*) as "PREDICT_CHURN?"
    from big_customers
''').show()

#### 4. Create a vectorized version of the UDF

For your reference, this was the original function we defined in the prior notebook

```
# Define function
# Note, the number and data types of the function arguments
#  match the columns in the DATA_SCIENCE_DB.NEW_DATA.CUSTOMERS
#  table.
def predict_churn(CUSTOMER_ID: str,           # Not used
                  SURNAME_MASKED: str,        # Not used
                  CREDIT_SCORE: int,
                  GEOGRAPHY: str,
                  GENDER: str,
                  AGE: int,
                  TENURE: str,
                  MILEAGE_POINTS: int,
                  NUM_OF_PRODUCTS: str,
                  HAS_AIRLINE_CREDIT_CARD: str,
                  IS_ACTIVE_MEMBER: str,
                  ESTIMATED_SALARY: str) -> bool:
    
    df_row = pd.DataFrame([locals()])
    
    model_file = sys._xoptions.get("snowflake_import_directory") + 'churn_classifier.joblib'

    with open(model_file,'rb') as f:
        model = joblib.load(f)
        return model.predict(df_row)

# Save as a Python UDF
session.clear_imports()
session.add_import('@MODEL_DATA/churn_classifier.joblib')
session.udf.register(name='predict_churn',
                     func=predict_churn,
                     is_permanent=True, 
                     stage_location='python_load',
                     replace = True)
```

In [ ]:
# Note the input datatype is now a PandasDataFrame, where each row of the 
# dataframe is a "vector" in the batch that will be passed in to this function
import joblib
def predict_churn_vec(df: PandasDataFrame[str, str, int, str, str, 
                                          int, str, int, str, str, 
                                          str, str]) -> PandasSeries[bool]:

    # We need to add names to the dataframe columns as our model expects to match 
    # these identifiers
    df.columns = ["CUSTOMER_ID", "SURNAME_MASKED", "CREDIT_SCORE", 
                  "GEOGRAPHY", "GENDER", "AGE", "TENURE", "MILEAGE_POINTS", 
                  "NUM_OF_PRODUCTS", "HAS_AIRLINE_CREDIT_CARD", 
                  "IS_ACTIVE_MEMBER", "ESTIMATED_SALARY"]
    
    ### Code Below is identical to Non-vectorized Python UDF
    # Notice the code between this line and the next comment are unchanged from 
    # the non-vectorized UDF
    model_file = sys._xoptions.get("snowflake_import_directory") + 'churn_classifier.joblib'

    with open(model_file,'rb') as f:
        model = joblib.load(f)
        return model.predict(df)
    ### Code Above is identical to Non-vectorized Python UDF

# Save as a Python UDF, same as with the non-vectorized Python UDF
session.clear_imports()
session.add_packages('snowflake-snowpark-python', 'scikit-learn', 'pandas', 'numpy')
session.add_import('@MODEL_DATA/churn_classifier.joblib')
session.udf.register(name='predict_churn_vec',
                     func=predict_churn_vec,
                     is_permanent=True, 
                     stage_location='python_load',
                     replace = True)

#### 5. Obtain timing with same batch, new Vectorized Python UDF

In [ ]:
%%time

# Let's see if using this new vectorized version of our scoring function is faster?
session.sql('''
    create or replace temporary table predictions_2 as
    select customer_id, 
           surname_masked,
           predict_churn_vec(*) as "PREDICT_CHURN?"
    from big_customers
''').show()

In [ ]:
# Stop your warehouse since we are done with it for now.
session.sql("ALTER WAREHOUSE " + session.get_current_warehouse() + " SUSPEND").collect()

## Summary

What we saw in this notebook was the potential performance benefit of vectorizing a Python UDF, specifically with a Data Science use-case.

Give some thoughts to why the dramatic difference in time?  See if you can spot some hints by looking at the respective query profiles for the two CTAS statements.  Hint: You will need to look at Step 2 of each profile.  Can you think of cases where the performance benefit would be less pronounced?

### In this notebook we did the following:

1. Connect to Snowflake
2. Prepare a larger set of customers to score
3. Establish baseline timing, non-vectorized Python UDF
4. Create a vectorized version of the UDF
5. Obtain timing with same batch, new Vectorized Python UDF